# Neo4j Schema Design & Graph Model

## Overview
Design the optimal Neo4j graph schema for the ESCO ecosystem. This notebook translates our relationship insights into a production-ready, query-optimized graph model.

## Goals
- Design node labels and properties for all ESCO entities
- Define relationship types and directions based on relationship insights
- Create Cypher constraints and indexes for performance
- Prototype import strategy with sample data
- Design queries that leverage our network analysis findings

In [9]:
import pandas as pd
import json
from pathlib import Path
import numpy as np

# Load insights from previous notebooks
DATA_RAW = Path('../data/raw')
DATA_INTERIM = Path('../data/interim')
DATA_SAMPLES = Path('../data/samples')

# Load relationship insights
with open(DATA_INTERIM / 'relationship_insights.json', 'r') as f:
    insights = json.load(f)

print("📊 Building on Previous Analysis:")
print(f"  • {insights['total_occupations']:,} occupations")
print(f"  • {insights['total_skills']:,} skills") 
print(f"  • {insights['total_relationships']:,} occupation-skill relationships")
print(f"  • {insights['collection_count']} skill collections")
print(f"  • Network: {insights['network_nodes']:,} nodes, {insights['network_edges']:,} edges")
print(f"  • Key insight: Language skills coverage only {insights['collection_coverage']['language_skills']:.1f}%")

📊 Building on Previous Analysis:
  • 3,039 occupations
  • 13,492 skills
  • 129,004 occupation-skill relationships
  • 6 skill collections
  • Network: 16,531 nodes, 128,987 edges
  • Key insight: Language skills coverage only 0.8%


## 1. Node Label Strategy

Define comprehensive node labels for all ESCO entity types with optimized properties based on our data analysis.

In [10]:
# Node label definitions with enhanced properties based on insights
node_schema = {
    'Occupation': {
        'description': 'ESCO occupations with ISCO classification',
        'properties': {
            'uri': 'STRING (UNIQUE)',
            'preferredLabel': 'STRING',
            'altLabels': 'STRING[]',
            'description': 'STRING',
            'iscoGroup': 'STRING',
            'status': 'STRING',
            'code': 'STRING',
            'inScheme': 'STRING',
            'regulatedProfessionNote': 'STRING',
            'scopeNote': 'STRING',
            'skillCount': 'INTEGER',  # Pre-computed for performance
            'essentialSkillCount': 'INTEGER',
            'optionalSkillCount': 'INTEGER'
        },
        'indexes': ['uri', 'preferredLabel', 'iscoGroup', 'code', 'skillCount']
    },
    
    'Skill': {
        'description': 'Core ESCO skills with type and reuse level',
        'properties': {
            'uri': 'STRING (UNIQUE)',
            'preferredLabel': 'STRING', 
            'altLabels': 'STRING[]',
            'description': 'STRING',
            'skillType': 'STRING',
            'reuseLevel': 'STRING',
            'status': 'STRING',
            'code': 'STRING',
            'inScheme': 'STRING',
            'scopeNote': 'STRING',
            'occupationCount': 'INTEGER',  # Pre-computed from degree analysis
            'essentialOccupationCount': 'INTEGER',
            'collections': 'STRING[]'  # Track multiple collection membership
        },
        'indexes': ['uri', 'preferredLabel', 'skillType', 'reuseLevel', 'occupationCount']
    },
    
    'SkillGroup': {
        'description': 'Skill groups and categories',
        'properties': {
            'uri': 'STRING (UNIQUE)',
            'preferredLabel': 'STRING',
            'altLabels': 'STRING[]', 
            'description': 'STRING',
            'code': 'STRING',
            'inScheme': 'STRING',
            'scopeNote': 'STRING',
            'hierarchyLevel': 'INTEGER',
            'childCount': 'INTEGER'
        },
        'indexes': ['uri', 'preferredLabel', 'hierarchyLevel']
    },
    
    'ISCOGroup': {
        'description': 'International Standard Classification of Occupations groups',
        'properties': {
            'uri': 'STRING (UNIQUE)',
            'preferredLabel': 'STRING',
            'code': 'STRING',
            'description': 'STRING',
            'inScheme': 'STRING',
            'occupationCount': 'INTEGER',
            'depth': 'INTEGER'
        },
        'indexes': ['uri', 'code', 'depth']
    }
}

# Enhanced collection-specific node labels based on coverage insights
collection_schema = {
    'DigitalSkill': {
        'extends': 'Skill',
        'description': 'Digital competence skills from DigComp framework',
        'additional_props': {
            'collection': 'STRING',
            'digCompArea': 'STRING',
            'coverageLevel': 'STRING'  # High coverage from insights
        }
    },
    
    'GreenSkill': {
        'extends': 'Skill', 
        'description': 'Skills related to environmental sustainability',
        'additional_props': {
            'collection': 'STRING',
            'sustainabilityDomain': 'STRING',
            'coverageLevel': 'STRING'  # High coverage from insights
        }
    },
    
    'LanguageSkill': {
        'extends': 'Skill',
        'description': 'Language and communication skills',
        'additional_props': {
            'collection': 'STRING',
            'languageFamily': 'STRING',
            'coverageLevel': 'STRING',  # Very low coverage - special handling
            'isUnderutilized': 'BOOLEAN'
        }
    },
    
    'TransversalSkill': {
        'extends': 'Skill',
        'description': 'Transferable skills across occupations',
        'additional_props': {
            'collection': 'STRING',
            'transversalCategory': 'STRING',
            'coverageLevel': 'STRING',  # Low coverage - special handling
            'isUnderutilized': 'BOOLEAN'
        }
    },
    
    'ResearchSkill': {
        'extends': 'Skill',
        'description': 'Research and analytical skills',
        'additional_props': {
            'collection': 'STRING',
            'researchDomain': 'STRING',
            'coverageLevel': 'STRING'  # Medium-high coverage
        }
    }
}

print("🏷️ Enhanced Node Label Strategy:")
for label, config in node_schema.items():
    print(f"  • {label}: {len(config['properties'])} properties, {len(config['indexes'])} indexes")

print(f"\n📚 Collection Labels: {len(collection_schema)} specialized skill types")
print("   Coverage-aware design based on relationship insights")

🏷️ Enhanced Node Label Strategy:
  • Occupation: 13 properties, 5 indexes
  • Skill: 13 properties, 5 indexes
  • SkillGroup: 9 properties, 3 indexes
  • ISCOGroup: 7 properties, 3 indexes

📚 Collection Labels: 5 specialized skill types
   Coverage-aware design based on relationship insights


## 2. Enhanced Relationship Type Design

Define relationship types with proper directions and properties for optimal traversal, incorporating our network analysis findings.

In [11]:
# Enhanced relationship type definitions based on insights
relationship_schema = {
    # Enhanced hierarchy relationships (separated by type for performance)
    'BROADER_THAN_SKILL': {
        'direction': 'OUTGOING',
        'description': 'Skill hierarchy relationships',
        'source': ['Skill', 'SkillGroup'],
        'target': ['Skill', 'SkillGroup'],
        'properties': {
            'hierarchyLevel': 'INTEGER',
            'isDirect': 'BOOLEAN',
            'depth': 'INTEGER'
        }
    },
    
    'BROADER_THAN_OCCUPATION': {
        'direction': 'OUTGOING', 
        'description': 'Occupation hierarchy relationships',
        'source': ['Occupation', 'ISCOGroup'],
        'target': ['Occupation', 'ISCOGroup'],
        'properties': {
            'hierarchyLevel': 'INTEGER',
            'isDirect': 'BOOLEAN',
            'depth': 'INTEGER'
        }
    },
    
    # Occupation-Skill relationships (optimized based on 52.4% essential / 47.6% optional split)
    'REQUIRES': {
        'direction': 'OUTGOING', 
        'description': 'Occupation requires specific skill (essential)',
        'source': ['Occupation'],
        'target': ['Skill', 'DigitalSkill', 'GreenSkill', 'LanguageSkill', 'TransversalSkill', 'ResearchSkill'],
        'properties': {
            'relationType': 'STRING',
            'skillType': 'STRING',
            'isEssential': 'BOOLEAN',
            'weight': 'FLOAT'  # For recommendation systems
        }
    },
    
    'OPTIONAL_SKILL': {
        'direction': 'OUTGOING',
        'description': 'Occupation optionally uses skill',
        'source': ['Occupation'], 
        'target': ['Skill', 'DigitalSkill', 'GreenSkill', 'LanguageSkill', 'TransversalSkill', 'ResearchSkill'],
        'properties': {
            'relationType': 'STRING',
            'skillType': 'STRING',
            'isEssential': 'BOOLEAN',
            'weight': 'FLOAT'  # For recommendation systems
        }
    },
    
    # Enhanced Skill-Skill relationships
    'HAS_PREREQUISITE': {
        'direction': 'OUTGOING',
        'description': 'Skill requires another skill as prerequisite',
        'source': ['Skill', 'DigitalSkill', 'GreenSkill', 'LanguageSkill', 'TransversalSkill', 'ResearchSkill'],
        'target': ['Skill', 'DigitalSkill', 'GreenSkill', 'LanguageSkill', 'TransversalSkill', 'ResearchSkill'],
        'properties': {
            'relationType': 'STRING',
            'dependencyLevel': 'STRING',
            'isStrong': 'BOOLEAN'
        }
    },
    
    'RELATED_TO': {
        'direction': 'UNDIRECTED',
        'description': 'Skills are related or complementary',
        'source': ['Skill', 'DigitalSkill', 'GreenSkill', 'LanguageSkill', 'TransversalSkill', 'ResearchSkill'],
        'target': ['Skill', 'DigitalSkill', 'GreenSkill', 'LanguageSkill', 'TransversalSkill', 'ResearchSkill'],
        'properties': {
            'relationType': 'STRING',
            'strength': 'FLOAT',
            'correlation': 'FLOAT'
        }
    },
    
    # Enhanced grouping relationships
    'PART_OF': {
        'direction': 'OUTGOING',
        'description': 'Entity belongs to a group or collection',
        'source': ['Occupation', 'Skill'],
        'target': ['SkillGroup', 'ISCOGroup'],
        'properties': {
            'membershipType': 'STRING',
            'isPrimary': 'BOOLEAN'
        }
    },
    
    # Enhanced collection relationships with coverage awareness
    'IN_COLLECTION': {
        'direction': 'OUTGOING',
        'description': 'Skill belongs to a specific collection with coverage metadata',
        'source': ['Skill'],
        'target': ['DigitalSkill', 'GreenSkill', 'LanguageSkill', 'TransversalSkill', 'ResearchSkill'],
        'properties': {
            'coverageLevel': 'STRING',  # high/medium/low based on insights
            'isActiveInNetwork': 'BOOLEAN',
            'occupationUsage': 'INTEGER'
        }
    },
    
    # New: Cross-collection relationships for underutilized skills
    'COMPLEMENTARY_TO': {
        'direction': 'UNDIRECTED',
        'description': 'Skills from different collections that work well together',
        'source': ['DigitalSkill', 'GreenSkill', 'LanguageSkill', 'TransversalSkill', 'ResearchSkill'],
        'target': ['DigitalSkill', 'GreenSkill', 'LanguageSkill', 'TransversalSkill', 'ResearchSkill'],
        'properties': {
            'synergyScore': 'FLOAT',
            'commonOccupations': 'INTEGER'
        }
    }
}

print("🔗 Enhanced Relationship Strategy:")
for rel_type, config in relationship_schema.items():
    sources = ', '.join(config['source'][:2]) + ('...' if len(config['source']) > 2 else '')
    targets = ', '.join(config['target'][:2]) + ('...' if len(config['target']) > 2 else '')
    print(f"  • {rel_type}: {sources} → {targets} ({config['direction']})")
    
print(f"\n🎯 Coverage-aware design for {insights['collection_coverage']['language_skills']:.1f}% language skills")

🔗 Enhanced Relationship Strategy:
  • BROADER_THAN_SKILL: Skill, SkillGroup → Skill, SkillGroup (OUTGOING)
  • BROADER_THAN_OCCUPATION: Occupation, ISCOGroup → Occupation, ISCOGroup (OUTGOING)
  • REQUIRES: Occupation → Skill, DigitalSkill... (OUTGOING)
  • OPTIONAL_SKILL: Occupation → Skill, DigitalSkill... (OUTGOING)
  • HAS_PREREQUISITE: Skill, DigitalSkill... → Skill, DigitalSkill... (OUTGOING)
  • RELATED_TO: Skill, DigitalSkill... → Skill, DigitalSkill... (UNDIRECTED)
  • PART_OF: Occupation, Skill → SkillGroup, ISCOGroup (OUTGOING)
  • IN_COLLECTION: Skill → DigitalSkill, GreenSkill... (OUTGOING)
  • COMPLEMENTARY_TO: DigitalSkill, GreenSkill... → DigitalSkill, GreenSkill... (UNDIRECTED)

🎯 Coverage-aware design for 0.8% language skills


## 3. Enhanced Constraint & Index Strategy

Design constraints and indexes for performance and data integrity, optimized for our specific dataset characteristics.

In [12]:
# Generate enhanced constraint statements
constraints = [
    # Unique constraints for URIs
    "CREATE CONSTRAINT occupation_uri IF NOT EXISTS FOR (o:Occupation) REQUIRE o.uri IS UNIQUE;",
    "CREATE CONSTRAINT skill_uri IF NOT EXISTS FOR (s:Skill) REQUIRE s.uri IS UNIQUE;",
    "CREATE CONSTRAINT skill_group_uri IF NOT EXISTS FOR (sg:SkillGroup) REQUIRE sg.uri IS UNIQUE;",
    "CREATE CONSTRAINT isco_group_uri IF NOT EXISTS FOR (ig:ISCOGroup) REQUIRE ig.uri IS UNIQUE;",
    
    # Collection unique constraints
    "CREATE CONSTRAINT digital_skill_uri IF NOT EXISTS FOR (ds:DigitalSkill) REQUIRE ds.uri IS UNIQUE;",
    "CREATE CONSTRAINT green_skill_uri IF NOT EXISTS FOR (gs:GreenSkill) REQUIRE gs.uri IS UNIQUE;",
    "CREATE CONSTRAINT language_skill_uri IF NOT EXISTS FOR (ls:LanguageSkill) REQUIRE ls.uri IS UNIQUE;",
    "CREATE CONSTRAINT transversal_skill_uri IF NOT EXISTS FOR (ts:TransversalSkill) REQUIRE ts.uri IS UNIQUE;",
    "CREATE CONSTRAINT research_skill_uri IF NOT EXISTS FOR (rs:ResearchSkill) REQUIRE rs.uri IS UNIQUE;"
]

# Generate enhanced index statements based on query patterns and data distribution
indexes = [
    # Occupation indexes
    "CREATE INDEX occupation_label IF NOT EXISTS FOR (o:Occupation) ON (o.preferredLabel);",
    "CREATE INDEX occupation_isco IF NOT EXISTS FOR (o:Occupation) ON (o.iscoGroup);",
    "CREATE INDEX occupation_code IF NOT EXISTS FOR (o:Occupation) ON (o.code);",
    "CREATE INDEX occupation_skill_count IF NOT EXISTS FOR (o:Occupation) ON (o.skillCount);",
    "CREATE INDEX occupation_isco_label IF NOT EXISTS FOR (o:Occupation) ON (o.iscoGroup, o.preferredLabel);",
    
    # Skill indexes
    "CREATE INDEX skill_label IF NOT EXISTS FOR (s:Skill) ON (s.preferredLabel);",
    "CREATE INDEX skill_type IF NOT EXISTS FOR (s:Skill) ON (s.skillType);",
    "CREATE INDEX skill_reuse_level IF NOT EXISTS FOR (s:Skill) ON (s.reuseLevel);",
    "CREATE INDEX skill_occupation_count IF NOT EXISTS FOR (s:Skill) ON (s.occupationCount);",
    "CREATE INDEX skill_type_label IF NOT EXISTS FOR (s:Skill) ON (s.skillType, s.preferredLabel);",
    "CREATE INDEX skill_collections IF NOT EXISTS FOR (s:Skill) ON (s.collections);",
    
    # SkillGroup indexes
    "CREATE INDEX skill_group_label IF NOT EXISTS FOR (sg:SkillGroup) ON (sg.preferredLabel);",
    "CREATE INDEX skill_group_level IF NOT EXISTS FOR (sg:SkillGroup) ON (sg.hierarchyLevel);",
    
    # ISCOGroup indexes  
    "CREATE INDEX isco_group_code IF NOT EXISTS FOR (ig:ISCOGroup) ON (ig.code);",
    "CREATE INDEX isco_group_depth IF NOT EXISTS FOR (ig:ISCOGroup) ON (ig.depth);",
    
    # Enhanced relationship indexes for performance
    "CREATE INDEX rel_requires IF NOT EXISTS FOR ()-[r:REQUIRES]-() ON (r.relationType, r.isEssential);",
    "CREATE INDEX rel_optional IF NOT EXISTS FOR ()-[r:OPTIONAL_SKILL]-() ON (r.relationType, r.isEssential);",
    "CREATE INDEX rel_broader_skill IF NOT EXISTS FOR ()-[r:BROADER_THAN_SKILL]-() ON (r.hierarchyLevel, r.isDirect);",
    "CREATE INDEX rel_broader_occupation IF NOT EXISTS FOR ()-[r:BROADER_THAN_OCCUPATION]-() ON (r.hierarchyLevel, r.isDirect);",
    "CREATE INDEX rel_collection IF NOT EXISTS FOR ()-[r:IN_COLLECTION]-() ON (r.coverageLevel, r.isActiveInNetwork);",
    
    # Full-text indexes for search
    "CREATE FULLTEXT INDEX occupation_search IF NOT EXISTS FOR (o:Occupation) ON EACH [o.preferredLabel, o.altLabels, o.description];",
    "CREATE FULLTEXT INDEX skill_search IF NOT EXISTS FOR (s:Skill) ON EACH [s.preferredLabel, s.altLabels, s.description];"
]

print("🗂️ Enhanced Constraint & Index Plan:")
print(f"  • {len(constraints)} unique constraints for URI integrity")
print(f"  • {len(indexes)} indexes for query performance")
print(f"  • Coverage: All core entities + collections + composite indexes + full-text search")
print(f"  • Optimized for: {insights['degree_stats']['mean']:.1f} avg degree, {insights['network_density']:.6f} density")

# Save constraint and index scripts
NEO4J_DIR = Path('../neo4j')
NEO4J_DIR.mkdir(exist_ok=True)

with open(NEO4J_DIR / 'constraints.cypher', 'w') as f:
    f.write('// ESCO Graph Constraints - Run First\n\n')
    f.write('\n'.join(constraints))

with open(NEO4J_DIR / 'indexes.cypher', 'w') as f:
    f.write('// ESCO Graph Indexes - Run After Constraints\n\n')
    f.write('\n'.join(indexes))

print(f"\n💾 Saved to: {NEO4J_DIR / 'constraints.cypher'}")
print(f"💾 Saved to: {NEO4J_DIR / 'indexes.cypher'}")

🗂️ Enhanced Constraint & Index Plan:
  • 9 unique constraints for URI integrity
  • 22 indexes for query performance
  • Coverage: All core entities + collections + composite indexes + full-text search
  • Optimized for: 15.6 avg degree, 0.000944 density

💾 Saved to: ..\neo4j\constraints.cypher
💾 Saved to: ..\neo4j\indexes.cypher


## 4. Enhanced Sample Data Prototyping

Test the schema design with a small sample dataset that represents our key insights.

In [13]:
# Create enhanced sample data for prototyping
def create_enhanced_sample_datasets():
    """Generate sample datasets for Neo4j import testing with coverage awareness"""
    
    # Load original data
    occupations = pd.read_csv(DATA_RAW / 'occupations_en.csv')
    skills = pd.read_csv(DATA_RAW / 'skills_en.csv')
    occupation_skills = pd.read_csv(DATA_RAW / 'occupationSkillRelations_en.csv')
    skill_groups = pd.read_csv(DATA_RAW / 'skillGroups_en.csv')
    
    # Load collections to ensure we sample across coverage levels
    digital_skills = pd.read_csv(DATA_RAW / 'digitalSkillsCollection_en.csv')
    language_skills = pd.read_csv(DATA_RAW / 'languageSkillsCollection_en.csv')
    
    # Create strategic samples that represent coverage insights
    sample_occupations = occupations.sample(frac=0.02, random_state=42)
    
    # Sample skills strategically: include high and low coverage collections
    digital_sample = digital_skills.sample(n=20, random_state=42)
    language_sample = language_skills.sample(n=10, random_state=42)  # Smaller sample for low coverage
    other_skills = skills[~skills['conceptUri'].isin(
        set(digital_skills['conceptUri']) | set(language_skills['conceptUri'])
    )].sample(frac=0.01, random_state=42)
    
    sample_skills = pd.concat([digital_sample, language_sample, other_skills])
    
    # Get related relationships
    sample_occ_uris = set(sample_occupations['conceptUri'])
    sample_skill_uris = set(sample_skills['conceptUri'])
    
    sample_occupation_skills = occupation_skills[
        occupation_skills['occupationUri'].isin(sample_occ_uris) & 
        occupation_skills['skillUri'].isin(sample_skill_uris)
    ]
    
    # Sample skill groups with hierarchy awareness
    sample_skill_groups = skill_groups.sample(frac=0.1, random_state=42)
    
    return {
        'occupations': sample_occupations,
        'skills': sample_skills,
        'occupation_skills': sample_occupation_skills,
        'skill_groups': sample_skill_groups,
        'digital_skills': digital_sample,
        'language_skills': language_sample
    }

# Generate enhanced samples
samples = create_enhanced_sample_datasets()

print("🧪 Enhanced Sample Data for Prototyping:")
for name, df in samples.items():
    print(f"  • {name}: {len(df):,} records")
    
print(f"\n🎯 Strategic sampling includes:")
print(f"  • {len(samples['digital_skills'])} digital skills (high coverage)")
print(f"  • {len(samples['language_skills'])} language skills (low coverage test)")

# Save sample data
DATA_SAMPLES.mkdir(exist_ok=True)
for name, df in samples.items():
    if name not in ['digital_skills', 'language_skills']:  # Don't save separate collection files
        df.to_csv(DATA_SAMPLES / f'{name}_sample.csv', index=False)

print(f"\n💾 Sample data saved to: {DATA_SAMPLES}/")

🧪 Enhanced Sample Data for Prototyping:
  • occupations: 61 records
  • skills: 153 records
  • occupation_skills: 26 records
  • skill_groups: 64 records
  • digital_skills: 20 records
  • language_skills: 10 records

🎯 Strategic sampling includes:
  • 20 digital skills (high coverage)
  • 10 language skills (low coverage test)

💾 Sample data saved to: ..\data\samples/
